# Day 082 — Exercise 1: Short-Term Working Memory

**What you'll build:** `WorkingMemory` — a bounded log of the recent turns in the current conversation.

**Why it matters:** an agent needs to remember what was *just* said to stay coherent across a few turns. Working memory is that scratchpad — but it's bounded (so the prompt never grows forever) and disposable (cleared when the session ends). Tomorrow's other half, long-term memory, is what actually persists.

In [ ]:
# Short-term working memory - a pure Python class, no imports needed.


## Task

`WorkingMemory(max_turns=10)`

- `add(role, content)` — append `{'role', 'content'}` (stringify content); if over `max_turns`, keep only the last `max_turns`; return `self`.
- `turns()` — a **copy** of the turn list.
- `render()` — one `role: content` line per turn.
- `clear()` — empty the log in place.
- `__len__` — number of turns held.

## Your Implementation

In [ ]:
class WorkingMemory:
    """Short-term, in-session memory: recent turns, bounded and clearable."""

    def __init__(self, max_turns=10):
        raise NotImplementedError

    def add(self, role, content):
        raise NotImplementedError

    def turns(self):
        raise NotImplementedError

    def render(self):
        raise NotImplementedError

    def clear(self):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError


In [ ]:

# ── short-term working memory (this session only) ────────────────────────────
class WorkingMemory:
    """Short-term memory: the recent turns of the current session.

    Bounded to the last `max_turns` messages so the prompt never grows without
    limit, and cleared at a session boundary. This is the agent's scratchpad -
    it does NOT survive a restart.
    """

    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self._turns = []

    def add(self, role, content):
        """Append a {role, content} turn; keep only the last max_turns. Returns self."""
        self._turns.append({"role": role, "content": str(content)})
        if len(self._turns) > self.max_turns:
            self._turns = self._turns[-self.max_turns:]
        return self

    def turns(self):
        """Return a copy of the recent turns."""
        return list(self._turns)

    def render(self):
        """Render the turns as text, one 'role: content' line each."""
        return "\n".join(t["role"] + ": " + t["content"] for t in self._turns)

    def clear(self):
        """Forget the session (end-of-session boundary)."""
        self._turns.clear()

    def __len__(self):
        return len(self._turns)


## Automated checks

In [ ]:

score, total = 0, 5
try:
    wm = WorkingMemory(max_turns=3)
    wm.add('user', 'hi').add('assistant', 'hello')
    assert len(wm) == 2
    score += 1; print("✅ add records turns and returns self (chainable)")

    assert wm.turns()[0] == {'role': 'user', 'content': 'hi'}
    score += 1; print("✅ turns() returns the {role, content} log")

    wm.turns().clear()
    assert len(wm) == 2
    score += 1; print("✅ turns() returns a copy, not the live list")

    for i in range(5):
        wm.add('user', str(i))
    assert len(wm) == 3 and wm.turns()[0]['content'] == '2'
    score += 1; print("✅ bounded to the last max_turns messages")

    assert 'user: hi' not in wm.render()      # 'hi' was evicted long ago
    wm.clear()
    assert len(wm) == 0
    score += 1; print("✅ render() formats the turns; clear() ends the session")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── short-term working memory (this session only) ────────────────────────────
class WorkingMemory:
    """Short-term memory: the recent turns of the current session.

    Bounded to the last `max_turns` messages so the prompt never grows without
    limit, and cleared at a session boundary. This is the agent's scratchpad -
    it does NOT survive a restart.
    """

    def __init__(self, max_turns=10):
        self.max_turns = max_turns
        self._turns = []

    def add(self, role, content):
        """Append a {role, content} turn; keep only the last max_turns. Returns self."""
        self._turns.append({"role": role, "content": str(content)})
        if len(self._turns) > self.max_turns:
            self._turns = self._turns[-self.max_turns:]
        return self

    def turns(self):
        """Return a copy of the recent turns."""
        return list(self._turns)

    def render(self):
        """Render the turns as text, one 'role: content' line each."""
        return "\n".join(t["role"] + ": " + t["content"] for t in self._turns)

    def clear(self):
        """Forget the session (end-of-session boundary)."""
        self._turns.clear()

    def __len__(self):
        return len(self._turns)
```

**Why bound the turns?** Every turn goes into the next prompt. Without a cap the context grows without limit — slower, costlier, and eventually past the model's window. Keeping the last `max_turns` is the simplest sliding-window policy that keeps recent context while staying bounded.

</details>